In [ ]:
import itertools
import json
import pathlib
import sys

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

sys.path.append(str(pathlib.PurePath('..')))
from linear_geodesic_optimization.mesh.rectangle import Mesh as RectangleMesh
from linear_geodesic_optimization.optimization.curvature import Computer as Curvature
from linear_geodesic_optimization.optimization.laplacian import Computer as Laplacian

In [ ]:
# Load up the outputted JSON blob
# path_output = pathlib.PurePath('..', '..', 'outputs', 'toy', 'test_grid_size', '0', '0.002_25_25', 'output.json')
path_output = pathlib.PurePath('..', '..', 'outputs', 'Internet2', 'sequential_long', '20240805060000', '0.002_50_50', 'output.json')
with open(path_output, 'r') as f:
    data_output = json.load(f)

parameters = data_output['parameters']

# Define the mesh from the optimization output
width = parameters['width']
height = parameters['height']
mesh_scale = parameters['mesh_scale']
mesh = RectangleMesh(width, height, mesh_scale)
mesh.set_parameters(data_output['final'] - np.array(data_output['initial']))

In [ ]:
# Grab the network data used to create the surface
graph_data, vertex_data, edge_data = data_output['network']
bounding_box = graph_data['bounding_box']
network_coordinates = graph_data['coordinates']
network_vertices = mesh.map_coordinates_to_support(np.array(network_coordinates), parameters['coordinates_scale'], bounding_box)
network_edges = graph_data['edges']
kappa_Ricci = edge_data['ricciCurvature']

In [ ]:
# Compute the curvature on the mesh
laplacian = Laplacian(mesh)
curvature = Curvature(mesh, laplacian)
curvature.forward()
kappa_Gaussian = curvature.kappa_G

In [ ]:
# Determine the balls around the network edges
fat_edges = mesh.get_fat_edges(
    network_vertices,
    network_edges,
    1.01 * 2**0.5 * mesh_scale / width
)

# For each fat edge, find the curvature at that point
kappa_Gaussian_groups = [
    [
        kappa_Gaussian[vertex.index]
        for vertex in fat_edge
    ]
    for fat_edge in fat_edges
]

# Also regroup the target curvatures by vertex index
kappa_Gaussian_by_vertex = [[] for _ in range(width * height)]
for target_curvature, fat_edge in zip(kappa_Ricci, fat_edges):
    for vertex in fat_edge:
        kappa_Gaussian_by_vertex[vertex.index].append(target_curvature)

In [ ]:
# Make lists of target curvatures and actual curvatures
xs, ys = zip(*itertools.chain(*[
    zip(itertools.repeat(x), y_group)
    for x, y_group in zip(kappa_Ricci, kappa_Gaussian_groups)
]))
xs = np.array(xs)
ys = np.array(ys)
ys_averaged = np.array([
    np.mean(y_group)
    for y_group in kappa_Gaussian_groups
])

In [ ]:
# Plot
fig, ax = plt.subplots(1, 1, dpi=150)
ax.plot([max(min(xs), min(ys)), min(max(xs), max(ys))], [max(min(xs), min(ys)), min(max(xs), max(ys))], 'r--')
ax.scatter(xs, ys, c='tab:blue', label='Individual')
ax.scatter(kappa_Ricci, ys_averaged, c='tab:orange', label='Averaged')
ax.set_xlabel('Target Curvature')
ax.set_ylabel('Actual Curvature')
ax.legend()
plt.show()

In [ ]:
# print(np.ma.masked_equal([[0, 1], [None, 3]], None))

# Heatmaps of curvatures
fig, (ax1, ax2) = plt.subplots(1, 2, dpi=150)

# Plot target curvatures
target_curvatures = np.array([
    np.mean(curvature_group) if curvature_group else np.nan
    for curvature_group in kappa_Gaussian_by_vertex
]).reshape((width, height))
cmap = plt.cm.RdBu.copy()
cmap.set_bad(color='gray')
ax1.imshow(target_curvatures.T, cmap=cmap, origin='lower', vmin=-1.2, vmax=1.2)
ax1.set_xticks([])
ax1.set_yticks([])

# Plot the actual curvatures
actual_curvatures = np.array(kappa_Gaussian).reshape((width, height))
actual_curvatures[np.isnan(target_curvatures)] = np.nan
im = ax2.imshow(actual_curvatures.T, cmap=cmap, origin='lower', vmin=-1.2, vmax=1.2)
ax2.set_xticks([])
ax2.set_yticks([])

divider1 = make_axes_locatable(ax1)
cax1 = divider1.append_axes('right', size='5%', pad=0.05)
cax1.set_visible(False)
divider2 = make_axes_locatable(ax2)
cax2 = divider2.append_axes('right', size='5%', pad=0.05)
fig.colorbar(im, cax=cax2)
plt.show()